# Notebook 02: Superposition and Measurement

---

**Author:** Milan Amrut Joshi  
**Date:** 2026-03-25  
**Prerequisites:** Notebook 01 (Qubits and Gates)  
**Framework:** Qiskit 1.x

---

## Table of Contents

1. The Superposition Principle
2. Mathematical Foundation of Superposition
3. The Hadamard Gate: Deep Dive
4. Multi-Qubit Superposition
5. The Measurement Postulate
6. Born Rule and Probability Amplitudes
7. Measurement in Different Bases
8. Statistical Verification Through Simulation
9. Quantum Interference
10. Exercises

## 1. The Superposition Principle

### Classical vs Quantum

In classical physics, a system is in **exactly one state** at any given time. A coin is either heads or tails. A light switch is either on or off.

In quantum mechanics, a system can be in a **superposition** — a linear combination of multiple states simultaneously.

```
Classical Coin:           Quantum "Coin" (Qubit):
                         
  ┌─────┐                  ┌──────────────────┐
  │HEADS│   OR             │ α·HEADS + β·TAILS│
  └─────┘                  │                  │
  ┌─────┐                  │ Both at once!    │
  │TAILS│                  │ Until measured.   │
  └─────┘                  └──────────────────┘
```

### Formal Statement

**Superposition Principle:** If $|\psi_1\rangle$ and $|\psi_2\rangle$ are valid quantum states, then any linear combination

$$|\psi\rangle = \alpha|\psi_1\rangle + \beta|\psi_2\rangle$$

is also a valid quantum state (provided $|\alpha|^2 + |\beta|^2 = 1$).

This is a consequence of the **linearity of quantum mechanics** — the Schrodinger equation is a linear differential equation, so any linear combination of solutions is also a solution.

### Important Clarification

Superposition does **NOT** mean:
- The qubit is in $|0\rangle$ AND $|1\rangle$ at the same time (that's a common but misleading pop-science description)
- The qubit is in $|0\rangle$ OR $|1\rangle$ and we just don't know which (that's a hidden variable interpretation, ruled out by Bell's theorem)

Superposition **means**:
- The qubit is in a state that has **no classical analog**
- It is described by a vector with complex amplitudes
- When measured, it **collapses** to one outcome with probabilities determined by the amplitudes

## 2. Mathematical Foundation of Superposition

### Single-Qubit Superposition

The general single-qubit state is:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}$$

where $\alpha, \beta \in \mathbb{C}$ and $|\alpha|^2 + |\beta|^2 = 1$.

### Degrees of Freedom

Since $\alpha$ and $\beta$ are complex numbers, we have 4 real parameters. But:
1. **Normalization** removes 1 degree of freedom: $|\alpha|^2 + |\beta|^2 = 1$
2. **Global phase** removes 1 more: $e^{i\gamma}|\psi\rangle$ and $|\psi\rangle$ are physically identical

This leaves **2 real parameters** — which is exactly the Bloch sphere angles $\theta$ and $\phi$:

$$|\psi\rangle = \cos\frac{\theta}{2}|0\rangle + e^{i\phi}\sin\frac{\theta}{2}|1\rangle$$

### Phase Matters!

Two states with the same probabilities but different **relative phases** are physically distinct:

$$|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle) \quad \text{and} \quad |-\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)$$

Both give 50/50 outcomes when measured in the computational basis ($|0\rangle, |1\rangle$), but they are **orthogonal states** — they can be perfectly distinguished by measuring in the $|+\rangle, |-\rangle$ basis.

This is the key insight: **relative phase is a quantum resource** that has no classical analog.

In [ ]:
# Demonstrate that phase matters: |+⟩ and |-⟩ look the same in Z-basis
# but different in X-basis
import numpy as np

ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)
ket_plus = (ket_0 + ket_1) / np.sqrt(2)
ket_minus = (ket_0 - ket_1) / np.sqrt(2)

print("=== Measurement Probabilities in Z-basis (|0⟩, |1⟩) ===")
print(f"|+⟩: P(0) = {abs(ket_plus[0])**2:.4f}, P(1) = {abs(ket_plus[1])**2:.4f}")
print(f"|-⟩: P(0) = {abs(ket_minus[0])**2:.4f}, P(1) = {abs(ket_minus[1])**2:.4f}")
print("→ Indistinguishable in Z-basis!")

print("\n=== Measurement Probabilities in X-basis (|+⟩, |-⟩) ===")
# To measure in X-basis, compute |⟨+|ψ⟩|² and |⟨-|ψ⟩|²
for name, state in [('|+⟩', ket_plus), ('|-⟩', ket_minus)]:
    p_plus = abs(np.vdot(ket_plus, state))**2
    p_minus = abs(np.vdot(ket_minus, state))**2
    print(f"{name}: P(+) = {p_plus:.4f}, P(-) = {p_minus:.4f}")
print("→ Perfectly distinguishable in X-basis!")

## 3. The Hadamard Gate: Deep Dive

The **Hadamard gate** is the most important single-qubit gate for creating superposition.

### Matrix Representation

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

### Action on Basis States

$$H|0\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}\begin{pmatrix} 1 \\ 0 \end{pmatrix} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix} = \frac{|0\rangle + |1\rangle}{\sqrt{2}} = |+\rangle$$

$$H|1\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}\begin{pmatrix} 0 \\ 1 \end{pmatrix} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ -1 \end{pmatrix} = \frac{|0\rangle - |1\rangle}{\sqrt{2}} = |-\rangle$$

### Compact Formula

$$H|x\rangle = \frac{1}{\sqrt{2}}\sum_{z \in \{0,1\}} (-1)^{xz} |z\rangle$$

For $x=0$: $H|0\rangle = \frac{1}{\sqrt{2}}((-1)^0|0\rangle + (-1)^0|1\rangle) = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$ ✓

For $x=1$: $H|1\rangle = \frac{1}{\sqrt{2}}((-1)^0|0\rangle + (-1)^1|1\rangle) = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$ ✓

### Properties of H

1. **Self-inverse:** $H^2 = I$ (Hadamard is its own inverse)
2. **Hermitian:** $H = H^\dagger$ (H equals its own adjoint)
3. **Unitary:** $H^\dagger H = I$
4. **Basis change:** H transforms between the Z-basis ($|0\rangle, |1\rangle$) and X-basis ($|+\rangle, |-\rangle$)

### H as a Basis Change

```
Z-basis          H          X-basis
  |0⟩    ─── H ───>   |+⟩
  |1⟩    ─── H ───>   |-⟩
  |+⟩    ─── H ───>   |0⟩
  |-⟩    ─── H ───>   |1⟩
```

The Hadamard gate is a **change of basis** matrix between the computational basis and the Hadamard basis.

In [ ]:
# Deep dive into the Hadamard gate
import numpy as np

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
I = np.eye(2)

ket_0 = np.array([1, 0])
ket_1 = np.array([0, 1])

print("=== Hadamard Matrix ===")
print(f"H = \n{np.round(H, 4)}")

print("\n=== H acting on basis states ===")
print(f"H|0⟩ = {np.round(H @ ket_0, 4)} = |+⟩")
print(f"H|1⟩ = {np.round(H @ ket_1, 4)} = |-⟩")

print("\n=== Key Properties ===")
print(f"H² = I? {np.allclose(H @ H, I)}")
print(f"H = H†? {np.allclose(H, H.conj().T)}")
print(f"H†H = I? {np.allclose(H.conj().T @ H, I)}")

# H = (X + Z) / sqrt(2)
X = np.array([[0, 1], [1, 0]])
Z = np.array([[1, 0], [0, -1]])
print(f"\nH = (X + Z)/√2? {np.allclose(H, (X + Z) / np.sqrt(2))}")

# Eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(H)
print(f"\n=== Eigenvalues of H ===")
print(f"λ₁ = {eigenvalues[0]:.4f}")
print(f"λ₂ = {eigenvalues[1]:.4f}")
print("(Eigenvalues are +1 and -1, as expected for a Hermitian unitary)")

## 4. Multi-Qubit Superposition

### Creating Uniform Superposition over n Qubits

Applying $H$ to each of $n$ qubits initialized to $|0\rangle$ creates a **uniform superposition** over all $2^n$ computational basis states:

$$H^{\otimes n}|0\rangle^{\otimes n} = \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n - 1} |x\rangle$$

### Example: 2 Qubits

$$(H \otimes H)|00\rangle = H|0\rangle \otimes H|0\rangle = |+\rangle \otimes |+\rangle$$

$$= \frac{1}{2}(|0\rangle + |1\rangle)(|0\rangle + |1\rangle) = \frac{1}{2}(|00\rangle + |01\rangle + |10\rangle + |11\rangle)$$

### Example: 3 Qubits

$$H^{\otimes 3}|000\rangle = \frac{1}{\sqrt{8}}(|000\rangle + |001\rangle + |010\rangle + |011\rangle + |100\rangle + |101\rangle + |110\rangle + |111\rangle)$$

Each state has probability $\frac{1}{8} = 12.5\%$.

### The Power of Superposition

```
n qubits    States in superposition    Classical equivalent
────────    ───────────────────────    ───────────────────
   1              2                    1 bit
   2              4                    2 bits
   10           1,024                  ~1 KB
   20         1,048,576                ~1 MB
   50      ~10^15                      ~1 PB
  100      ~10^30                      More than atoms in the universe
  300      ~10^90                      Exceeds particles in observable universe
```

This exponential scaling is the source of quantum computing's potential power.

In [ ]:
# Create multi-qubit superposition with Qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# 1-qubit superposition
qc1 = QuantumCircuit(1)
qc1.h(0)
sv1 = Statevector.from_instruction(qc1)
print("=== 1-Qubit Superposition ===")
print(f"State: {sv1}")
print(f"Probabilities: {sv1.probabilities_dict()}")

# 2-qubit superposition
qc2 = QuantumCircuit(2)
qc2.h(0)
qc2.h(1)
sv2 = Statevector.from_instruction(qc2)
print("\n=== 2-Qubit Superposition ===")
print(f"State: {sv2}")
print(f"Probabilities: {sv2.probabilities_dict()}")

# 3-qubit superposition
qc3 = QuantumCircuit(3)
qc3.h(0)
qc3.h(1)
qc3.h(2)
sv3 = Statevector.from_instruction(qc3)
print("\n=== 3-Qubit Superposition ===")
print(f"State: {sv3}")
print(f"Probabilities: {sv3.probabilities_dict()}")
print(f"Each of the {2**3} states has probability {1/2**3:.4f}")

In [ ]:
# Visualize the circuits
from qiskit import QuantumCircuit

# 3-qubit uniform superposition circuit
qc = QuantumCircuit(3)
qc.h(0)
qc.h(1)
qc.h(2)

print("3-Qubit Uniform Superposition Circuit:")
print(qc.draw('text'))

# 4-qubit version
qc4 = QuantumCircuit(4)
for i in range(4):
    qc4.h(i)

print("\n4-Qubit Uniform Superposition Circuit:")
print(qc4.draw('text'))
print(f"\nThis creates a superposition of {2**4} = 16 states!")

## 5. The Measurement Postulate

### Postulate (von Neumann Measurement)

When we measure a quantum state $|\psi\rangle = \sum_i \alpha_i |i\rangle$ in the computational basis:

1. **Outcome:** We get result $|i\rangle$ with probability $p_i = |\alpha_i|^2$
2. **Collapse:** After measurement, the state **collapses** to $|i\rangle$
3. **Irreversibility:** The original superposition is destroyed — you cannot recover $|\psi\rangle$

### Measurement as Projection

Measurement in the computational basis is described by the **projection operators**:

$$P_0 = |0\rangle\langle 0| = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}, \quad P_1 = |1\rangle\langle 1| = \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix}$$

The probability of outcome $i$ is:

$$p_i = \langle\psi|P_i|\psi\rangle = |\langle i|\psi\rangle|^2$$

After getting outcome $i$, the post-measurement state is:

$$|\psi'\rangle = \frac{P_i|\psi\rangle}{\sqrt{p_i}}$$

### Example

Let $|\psi\rangle = \frac{\sqrt{3}}{2}|0\rangle + \frac{1}{2}|1\rangle$.

- $P(0) = |\frac{\sqrt{3}}{2}|^2 = \frac{3}{4} = 75\%$
- $P(1) = |\frac{1}{2}|^2 = \frac{1}{4} = 25\%$
- $P(0) + P(1) = 1$ ✓

```
Before Measurement:          After Measurement:

  |ψ⟩ = √3/2|0⟩ + 1/2|1⟩    75% chance → |0⟩
                               25% chance → |1⟩
                               
  (superposition)              (definite state)
```

In [ ]:
# Demonstrate measurement and collapse
import numpy as np

# State: |ψ⟩ = √3/2 |0⟩ + 1/2 |1⟩
alpha = np.sqrt(3) / 2
beta = 1 / 2
psi = np.array([alpha, beta], dtype=complex)

print(f"|ψ⟩ = {alpha:.4f}|0⟩ + {beta:.4f}|1⟩")
print(f"\nP(0) = |α|² = {abs(alpha)**2:.4f} = {abs(alpha)**2*100:.1f}%")
print(f"P(1) = |β|² = {abs(beta)**2:.4f} = {abs(beta)**2*100:.1f}%")
print(f"P(0) + P(1) = {abs(alpha)**2 + abs(beta)**2:.4f}")

# Simulate measurements
np.random.seed(42)
n_measurements = 10000
outcomes = np.random.choice([0, 1], size=n_measurements, p=[abs(alpha)**2, abs(beta)**2])
count_0 = np.sum(outcomes == 0)
count_1 = np.sum(outcomes == 1)

print(f"\n=== Simulating {n_measurements} measurements ===")
print(f"  |0⟩: {count_0} times ({count_0/n_measurements*100:.1f}%) — expected 75%")
print(f"  |1⟩: {count_1} times ({count_1/n_measurements*100:.1f}%) — expected 25%")

# Projection operators
P0 = np.array([[1, 0], [0, 0]], dtype=complex)
P1 = np.array([[0, 0], [0, 1]], dtype=complex)

# Compute probabilities via projection
p0 = np.real(psi.conj() @ P0 @ psi)
p1 = np.real(psi.conj() @ P1 @ psi)
print(f"\n=== Via Projection Operators ===")
print(f"  ⟨ψ|P₀|ψ⟩ = {p0:.4f}")
print(f"  ⟨ψ|P₁|ψ⟩ = {p1:.4f}")

## 6. Born Rule and Probability Amplitudes

### The Born Rule

The **Born rule** (Max Born, 1926) is the fundamental link between quantum states and measurable probabilities:

$$P(\text{outcome } i) = |\langle i | \psi \rangle|^2 = |\alpha_i|^2$$

The **probability amplitude** $\alpha_i = \langle i|\psi\rangle$ is a complex number. The probability is its **modulus squared**.

### Why Complex Amplitudes?

The amplitude $\alpha = |\alpha|e^{i\phi}$ has:
- **Magnitude** $|\alpha|$: determines probability ($|\alpha|^2$)
- **Phase** $\phi$: determines interference effects

```
Complex amplitude α = |α|e^(iφ)

  Imaginary
      ↑
      │    ╱ α
      │   ╱
      │  ╱ |α|
      │ ╱
      │╱ φ
  ────┼──────→  Real
      │
      
  Probability = |α|²
  Phase φ affects interference
```

### Multi-Qubit Born Rule

For an $n$-qubit state $|\psi\rangle = \sum_{x=0}^{2^n-1} \alpha_x |x\rangle$:

$$P(\text{measuring } |x\rangle) = |\alpha_x|^2$$

$$\sum_{x=0}^{2^n-1} |\alpha_x|^2 = 1$$

### Partial Measurement

If we have 2 qubits in state $|\psi\rangle = \alpha_{00}|00\rangle + \alpha_{01}|01\rangle + \alpha_{10}|10\rangle + \alpha_{11}|11\rangle$ and measure only the **first qubit**:

- $P(\text{first qubit} = 0) = |\alpha_{00}|^2 + |\alpha_{01}|^2$
- $P(\text{first qubit} = 1) = |\alpha_{10}|^2 + |\alpha_{11}|^2$

After measuring first qubit = 0, the state collapses to:

$$|\psi'\rangle = \frac{\alpha_{00}|00\rangle + \alpha_{01}|01\rangle}{\sqrt{|\alpha_{00}|^2 + |\alpha_{01}|^2}}$$

In [ ]:
# Born rule demonstration with various states
import numpy as np

print("=== Born Rule: P(i) = |⟨i|ψ⟩|² ===")

# State 1: Equal superposition
psi1 = np.array([1, 1], dtype=complex) / np.sqrt(2)
print(f"\n|+⟩ = (|0⟩ + |1⟩)/√2")
print(f"  P(0) = |1/√2|² = {abs(psi1[0])**2:.4f}")
print(f"  P(1) = |1/√2|² = {abs(psi1[1])**2:.4f}")

# State 2: Unequal superposition
psi2 = np.array([np.sqrt(0.3), np.sqrt(0.7)], dtype=complex)
print(f"\n|ψ₂⟩ = √0.3|0⟩ + √0.7|1⟩")
print(f"  P(0) = {abs(psi2[0])**2:.4f}")
print(f"  P(1) = {abs(psi2[1])**2:.4f}")

# State 3: Complex amplitudes
psi3 = np.array([1+1j, 1-1j], dtype=complex) / 2
print(f"\n|ψ₃⟩ = (1+i)/2 |0⟩ + (1-i)/2 |1⟩")
print(f"  α₀ = {psi3[0]},  |α₀|² = {abs(psi3[0])**2:.4f}")
print(f"  α₁ = {psi3[1]},  |α₁|² = {abs(psi3[1])**2:.4f}")
print(f"  Sum = {abs(psi3[0])**2 + abs(psi3[1])**2:.4f}")

# State 4: phase doesn't affect probabilities
psi4a = np.array([1, 1], dtype=complex) / np.sqrt(2)
psi4b = np.array([1, -1], dtype=complex) / np.sqrt(2)
psi4c = np.array([1, 1j], dtype=complex) / np.sqrt(2)

print(f"\n=== Phase Doesn't Affect Z-basis Probabilities ===")
for name, state in [("|+⟩", psi4a), ("|-⟩", psi4b), ("|i⟩", psi4c)]:
    print(f"  {name}: P(0) = {abs(state[0])**2:.4f}, P(1) = {abs(state[1])**2:.4f}")

## 7. Measurement in Different Bases

So far we've measured in the **computational basis** (Z-basis): $\{|0\rangle, |1\rangle\}$. But we can measure in any orthonormal basis!

### Common Measurement Bases

| Basis | States | How to measure |
|-------|--------|----------------|
| Z-basis (computational) | $|0\rangle, |1\rangle$ | Direct measurement |
| X-basis (Hadamard) | $|+\rangle, |-\rangle$ | Apply $H$ then measure in Z |
| Y-basis | $|R\rangle, |L\rangle$ | Apply $S^\dagger H$ then measure in Z |

### The Trick: Change of Basis

To measure in basis $\{|b_0\rangle, |b_1\rangle\}$:
1. Apply the unitary $U$ that maps $|b_0\rangle \to |0\rangle$ and $|b_1\rangle \to |1\rangle$
2. Measure in the computational basis

```
Measuring in X-basis:

  |ψ⟩ ──── H ──── Measure in Z ────
            │
            └── This converts X-basis to Z-basis
                 |+⟩ → |0⟩
                 |-⟩ → |1⟩
```

In [ ]:
# Measurement in different bases with Qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt

simulator = AerSimulator()
shots = 4000

# Prepare |+⟩ state and measure in Z-basis vs X-basis
print("=== Measuring |+⟩ = (|0⟩+|1⟩)/√2 ===")

# Z-basis measurement of |+⟩
qc_z = QuantumCircuit(1, 1)
qc_z.h(0)          # Prepare |+⟩
qc_z.measure(0, 0)  # Measure in Z-basis

result_z = simulator.run(qc_z, shots=shots).result()
counts_z = result_z.get_counts()
print(f"Z-basis measurement: {counts_z}")
print("→ Random 50/50 outcomes (|+⟩ is superposition in Z-basis)")

# X-basis measurement of |+⟩
qc_x = QuantumCircuit(1, 1)
qc_x.h(0)          # Prepare |+⟩
qc_x.h(0)          # Change to X-basis (H maps |+⟩→|0⟩)
qc_x.measure(0, 0)  # Measure in Z-basis (effectively X-basis)

result_x = simulator.run(qc_x, shots=shots).result()
counts_x = result_x.get_counts()
print(f"\nX-basis measurement: {counts_x}")
print("→ Always |0⟩ (|+⟩ is an eigenstate of X-basis measurement)")

# Now measure |-⟩ in X-basis
qc_xm = QuantumCircuit(1, 1)
qc_xm.x(0)          # |1⟩
qc_xm.h(0)          # Prepare |-⟩
qc_xm.h(0)          # Change to X-basis (H maps |-⟩→|1⟩)
qc_xm.measure(0, 0)

result_xm = simulator.run(qc_xm, shots=shots).result()
counts_xm = result_xm.get_counts()
print(f"\nX-basis measurement of |-⟩: {counts_xm}")
print("→ Always |1⟩ (|-⟩ maps to |1⟩ in X-basis)")

## 8. Statistical Verification Through Simulation

One of the key features of quantum mechanics is its **probabilistic nature**. Let's verify the Born rule by running many measurements and comparing the observed frequencies to the predicted probabilities.

### Experiment Design

We will:
1. Prepare a state with known, non-trivial probabilities
2. Measure it many times (100, 1000, 10000 shots)
3. Plot histograms and verify convergence to Born rule predictions
4. Analyze statistical fluctuations

In [ ]:
# Statistical verification: measure a superposition state many times
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt
import numpy as np

# Prepare state: Ry(2*arccos(sqrt(0.7))) |0⟩ gives P(0) = 0.7, P(1) = 0.3
theta = 2 * np.arccos(np.sqrt(0.7))  # Ry rotation angle

simulator = AerSimulator()
shot_counts = [100, 500, 1000, 5000, 10000, 50000]

print(f"Target probabilities: P(0) = 0.70, P(1) = 0.30")
print(f"Ry rotation angle: θ = {theta:.4f} radians\n")

results = []
for shots in shot_counts:
    qc = QuantumCircuit(1, 1)
    qc.ry(theta, 0)
    qc.measure(0, 0)
    
    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts()
    
    n0 = counts.get('0', 0)
    n1 = counts.get('1', 0)
    p0_obs = n0 / shots
    p1_obs = n1 / shots
    error = abs(p0_obs - 0.7)
    
    results.append((shots, p0_obs, p1_obs, error))
    print(f"Shots: {shots:>6} | P(0) = {p0_obs:.4f} | P(1) = {p1_obs:.4f} | Error = {error:.4f}")

print("\n→ As shots increase, observed frequencies converge to Born rule predictions!")
print("  This is the law of large numbers applied to quantum measurements.")

In [ ]:
# Plot histogram of measurement results
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt
import numpy as np

simulator = AerSimulator()

# Create the H|0⟩ = |+⟩ state and measure 10000 times
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

shots = 10000
result = simulator.run(qc, shots=shots).result()
counts = result.get_counts()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
labels = sorted(counts.keys())
values = [counts[k] for k in labels]
expected = [shots/2, shots/2]

x = np.arange(len(labels))
width = 0.35

axes[0].bar(x - width/2, values, width, label='Observed', color='steelblue', alpha=0.8)
axes[0].bar(x + width/2, expected, width, label='Expected', color='coral', alpha=0.8)
axes[0].set_xlabel('Measurement Outcome', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title(f'H|0⟩ Measurement ({shots} shots)', fontsize=14)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, fontsize=12)
axes[0].legend(fontsize=11)
axes[0].axhline(y=shots/2, color='gray', linestyle='--', alpha=0.5)

# Convergence plot: run multiple experiments
shot_range = [10, 50, 100, 200, 500, 1000, 2000, 5000, 10000]
p0_values = []
for s in shot_range:
    r = simulator.run(qc, shots=s).result()
    c = r.get_counts()
    p0_values.append(c.get('0', 0) / s)

axes[1].plot(shot_range, p0_values, 'bo-', markersize=6, label='Observed P(0)')
axes[1].axhline(y=0.5, color='red', linestyle='--', label='Expected P(0) = 0.5')
axes[1].set_xscale('log')
axes[1].set_xlabel('Number of Shots (log scale)', fontsize=12)
axes[1].set_ylabel('P(0)', fontsize=12)
axes[1].set_title('Convergence to Born Rule', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].set_ylim([0.3, 0.7])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Quantum Interference

**Interference** is what makes quantum computing powerful. It is the phenomenon where probability amplitudes **add or cancel** depending on their relative phase.

### Constructive vs Destructive Interference

Consider two paths with amplitudes $\alpha_1$ and $\alpha_2$ leading to the same outcome:

- **Constructive:** If $\alpha_1$ and $\alpha_2$ have the same phase, $|\alpha_1 + \alpha_2|^2 > |\alpha_1|^2 + |\alpha_2|^2$ — probability increases
- **Destructive:** If they have opposite phase, $|\alpha_1 + \alpha_2|^2 < |\alpha_1|^2 + |\alpha_2|^2$ — probability decreases

```
Constructive:                  Destructive:
  α₁ = 1/√2                    α₁ =  1/√2
  α₂ = 1/√2                    α₂ = -1/√2
  ─────────                     ─────────
  α₁ + α₂ = √2                 α₁ + α₂ = 0
  P = |√2|² = 2  (enhanced!)   P = |0|² = 0  (cancelled!)
  
  (must renormalize)            (probability = 0!)
```

### The Mach-Zehnder Interferometer (Quantum Circuit Analog)

The classic demonstration of interference in quantum circuits is $H \circ H = I$:

```
|0⟩ ─── H ─── H ─── Measure
         │     │
         │     └─ Second H: paths interfere
         └─ First H: creates two paths
         
Path 1: |0⟩ → |0⟩ → |0⟩  amplitude: 1/√2 × 1/√2 = 1/2
Path 2: |0⟩ → |1⟩ → |0⟩  amplitude: 1/√2 × 1/√2 = 1/2
                                                    ───
                                    Total for |0⟩:  1   ← constructive!

Path 3: |0⟩ → |0⟩ → |1⟩  amplitude:  1/√2 × 1/√2 =  1/2
Path 4: |0⟩ → |1⟩ → |1⟩  amplitude:  1/√2 × (-1/√2) = -1/2
                                                      ───
                                    Total for |1⟩:    0   ← destructive!
```

Result: 100% probability of measuring $|0\rangle$ — interference perfectly steers the outcome!

In [ ]:
# Demonstrate quantum interference: H-H = I
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

simulator = AerSimulator()

# Single H: creates superposition (no interference yet)
qc1 = QuantumCircuit(1, 1)
qc1.h(0)
qc1.measure(0, 0)

# Double H: interference recovers |0⟩
qc2 = QuantumCircuit(1, 1)
qc2.h(0)
qc2.h(0)
qc2.measure(0, 0)

# Triple H: same as single H
qc3 = QuantumCircuit(1, 1)
qc3.h(0)
qc3.h(0)
qc3.h(0)
qc3.measure(0, 0)

shots = 4000
r1 = simulator.run(qc1, shots=shots).result().get_counts()
r2 = simulator.run(qc2, shots=shots).result().get_counts()
r3 = simulator.run(qc3, shots=shots).result().get_counts()

print("=== Quantum Interference Demo ===")
print(f"\n1× Hadamard (superposition):       {r1}")
print(f"2× Hadamard (interference → |0⟩):   {r2}")
print(f"3× Hadamard (superposition again):   {r3}")

print("\n→ Even number of H gates: constructive interference → |0⟩ with certainty")
print("→ Odd number of H gates: equal superposition")

In [ ]:
# Interference with phase: H-Z-H gives different result than H-H
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

simulator = AerSimulator()
shots = 4000

# H-H: constructive interference → |0⟩
qc_hh = QuantumCircuit(1, 1)
qc_hh.h(0)
qc_hh.h(0)
qc_hh.measure(0, 0)

# H-Z-H: Z flips the phase, causing destructive interference → |1⟩!
qc_hzh = QuantumCircuit(1, 1)
qc_hzh.h(0)   # |0⟩ → |+⟩
qc_hzh.z(0)   # |+⟩ → |-⟩  (phase flip!)
qc_hzh.h(0)   # |-⟩ → |1⟩  (destructive interference for |0⟩)
qc_hzh.measure(0, 0)

# H-S-H: partial phase shift → mixed interference
qc_hsh = QuantumCircuit(1, 1)
qc_hsh.h(0)   # |0⟩ → |+⟩
qc_hsh.s(0)   # Adds π/2 phase
qc_hsh.h(0)
qc_hsh.measure(0, 0)

r_hh = simulator.run(qc_hh, shots=shots).result().get_counts()
r_hzh = simulator.run(qc_hzh, shots=shots).result().get_counts()
r_hsh = simulator.run(qc_hsh, shots=shots).result().get_counts()

print("=== Phase Controls Interference ===")
print(f"\nH-H   (no phase):     {r_hh}  → always |0⟩ (constructive)")
print(f"H-Z-H (π phase):      {r_hzh}  → always |1⟩ (destructive for |0⟩)")
print(f"H-S-H (π/2 phase):    {r_hsh}  → 50/50 (partial interference)")

# Show the statevectors
for name, ops in [('H-H', ['h', 'h']), ('H-Z-H', ['h', 'z', 'h']), ('H-S-H', ['h', 's', 'h'])]:
    qc = QuantumCircuit(1)
    for op in ops:
        getattr(qc, op)(0)
    sv = Statevector.from_instruction(qc)
    print(f"\n{name} statevector: {sv}")

In [ ]:
# Comprehensive interference visualization
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt
import numpy as np

# Vary the phase from 0 to 2π and observe interference
phases = np.linspace(0, 2*np.pi, 100)
p0_values = []
p1_values = []

for phi in phases:
    qc = QuantumCircuit(1)
    qc.h(0)       # Create superposition
    qc.rz(phi, 0) # Apply phase
    qc.h(0)       # Interfere
    
    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities()
    p0_values.append(probs[0])
    p1_values.append(probs[1])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(phases, p0_values, 'b-', linewidth=2, label='P(0)')
ax.plot(phases, p1_values, 'r--', linewidth=2, label='P(1)')

# Mark special points
special_phases = {'φ=0\n(H-H=I)': 0, 'φ=π/2\n(H-S-H)': np.pi/2, 
                  'φ=π\n(H-Z-H)': np.pi, 'φ=3π/2': 3*np.pi/2, 'φ=2π': 2*np.pi}
for label, phi in special_phases.items():
    idx = np.argmin(np.abs(phases - phi))
    ax.annotate(label, (phases[idx], p0_values[idx]), 
                textcoords="offset points", xytext=(0, 15), ha='center', fontsize=9)
    ax.plot(phases[idx], p0_values[idx], 'ko', markersize=6)

ax.set_xlabel('Phase φ (radians)', fontsize=13)
ax.set_ylabel('Probability', fontsize=13)
ax.set_title('Quantum Interference: H - Rz(φ) - H', fontsize=15, fontweight='bold')
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim([-0.05, 1.05])
plt.tight_layout()
plt.show()

print("This plot shows how the phase φ controls the interference pattern.")
print("P(0) = cos²(φ/2), P(1) = sin²(φ/2) — a perfect sinusoidal pattern!")

## 10. Exercises

### Exercise 1: Custom Superposition
Create a state with $P(0) = 0.85$ and $P(1) = 0.15$ using the $R_y$ gate. Verify with 10,000 shots.

### Exercise 2: Multi-Qubit Measurement
Create a 3-qubit circuit where:
- Qubit 0 is in state $|+\rangle$
- Qubit 1 is in state $|0\rangle$
- Qubit 2 is in state $|1\rangle$

Predict the measurement probabilities for all 8 outcomes, then verify with simulation.

### Exercise 3: Interference Pattern
Implement the circuit $H - T^n - H$ for $n = 0, 1, 2, ..., 8$ and plot $P(0)$ vs $n$. The T gate adds $\pi/4$ phase each time, so you should see $P(0) = \cos^2(n\pi/8)$.

### Exercise 4: Repeated Measurement
Prepare $|+\rangle$, measure it, and then immediately measure again. Show that the second measurement always gives the same result as the first (demonstrating wave function collapse).

### Exercise 5: Statistical Analysis
Prepare the state $|\psi\rangle = \sqrt{0.3}|0\rangle + \sqrt{0.7}|1\rangle$. Run the measurement experiment with 100, 500, 1000, 5000, and 10000 shots. Calculate the standard deviation of the observed $P(0)$ and compare with the theoretical value $\sigma = \sqrt{p(1-p)/n}$.

In [ ]:
# Exercise 1 Solution: Custom superposition with P(0) = 0.85
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

# Ry(θ)|0⟩ = cos(θ/2)|0⟩ + sin(θ/2)|1⟩
# We want cos²(θ/2) = 0.85, so θ = 2*arccos(√0.85)
p_target = 0.85
theta = 2 * np.arccos(np.sqrt(p_target))
print(f"For P(0) = {p_target}, θ = {theta:.6f} radians")

qc = QuantumCircuit(1, 1)
qc.ry(theta, 0)
qc.measure(0, 0)

simulator = AerSimulator()
result = simulator.run(qc, shots=10000).result()
counts = result.get_counts()

n0 = counts.get('0', 0)
n1 = counts.get('1', 0)
print(f"\nResults (10000 shots):")
print(f"  |0⟩: {n0} ({n0/10000:.4f}) — target: {p_target}")
print(f"  |1⟩: {n1} ({n1/10000:.4f}) — target: {1-p_target}")

## Summary

In this notebook, we covered:

1. **Superposition principle** — quantum states can be linear combinations of basis states
2. **Hadamard gate** — creates equal superposition; self-inverse; changes between Z and X bases
3. **Multi-qubit superposition** — $n$ Hadamard gates create uniform superposition over $2^n$ states
4. **Measurement postulate** — measurement collapses the state; outcomes are probabilistic
5. **Born rule** — probability = $|\text{amplitude}|^2$; verified by repeated measurements
6. **Measurement in different bases** — apply basis-change unitary before Z-measurement
7. **Quantum interference** — amplitudes add/cancel based on relative phase; this is the key to quantum algorithms

### Key Takeaway

Quantum algorithms work by carefully arranging **constructive interference** on correct answers and **destructive interference** on wrong answers. This is the fundamental mechanism behind quantum speedup.

### Next Notebook

In **Notebook 03**, we explore **quantum entanglement** — the other key quantum resource, including Bell states, EPR paradox, and Bell inequalities.